In [1]:
import pandas as pd

In [2]:
reproductibiliy_path = "dynamicTasteDistortion/reproductibility_data"

## Tuning related figures/tables

### Oracle model ones

In [3]:
globo_path = "dynamicTasteDistortion/artifacts/results/globo_1m/oracle_model_f1_results.pkl"

In [4]:
results = pd.read_pickle(globo_path).sort_values(by="test_f1", ascending=False)

In [5]:
results

,model_name,params,validation_f1,test_f1,is_winning_variant
22,SVD,"{'n_factors': 15, 'n_epochs': 10, 'lr_all': 0....",0.657771,0.0,True
51,SVD++,"{'n_factors': 15, 'n_epochs': 10, 'lr_all': 0....",0.658021,0.0,True
76,NMF,"{'n_factors': 15, 'n_epochs': 100, 'reg_pu': 0...",0.634086,0.0,True
0,SVD,"{'n_factors': 15, 'n_epochs': 30, 'lr_all': 0....",0.648807,NaN,False
1,SVD,"{'n_factors': 30, 'n_epochs': 30, 'lr_all': 0....",0.654285,NaN,False
...,...,...,...,...,...
80,NMF,"{'n_factors': 30, 'n_epochs': 100, 'reg_pu': 0...",0.625803,NaN,False
81,NMF,"{'n_factors': 100, 'n_epochs': 100, 'reg_pu': ...",0.623483,NaN,False
82,NMF,"{'n_factors': 100, 'n_epochs': 50, 'reg_pu': 0...",0.629535,NaN,False
83,NMF,"{'n_factors': 100, 'n_epochs': 100, 'reg_pu': ...",0.626395,NaN,False


In [6]:
results.to_pickle(f"{reproductibiliy_path}/globo_oracle_full_results.pkl")


In [7]:
results.to_csv(f"{reproductibiliy_path}/globo_oracle_full_results.csv")

### BPR Tuning related data

In [8]:
globo = pd.read_csv(
    "dynamicTasteDistortion/artifacts/model/bpr_globo_1m_n_users=1000_cv_results.csv"
).sort_values(by="map", ascending=False)

In [9]:
globo

,Unnamed: 0,factors,lr,reg_lambda,num_negatives,n_epochs,map
0,5,64,0.00010,0.00100,5,7,0.133950
1,0,64,0.01000,0.00001,5,5,0.133203
2,18,32,0.01000,0.00001,5,10,0.133073
3,9,64,0.01000,0.01000,5,7,0.132870
4,17,32,0.00100,0.00010,5,10,0.132343
5,8,16,0.00010,0.00010,5,10,0.131370
6,16,16,0.00100,0.01000,1,5,0.130960
7,3,128,0.00010,0.00010,1,7,0.130740
8,19,32,0.00001,0.01000,10,10,0.129597
9,1,32,0.00010,0.00010,5,7,0.128840


In [10]:
globo.to_pickle(f"{reproductibiliy_path}/globo_bpr_full_results.pkl")
globo.to_csv(f"{reproductibiliy_path}/globo_bpr_full_results.csv")

## Results related figures


In [28]:
from dynamicTasteDistortion.ioUtils import read_experiment, read_metrics,read_metrics_per_seed, get_experiment_artifacts_path, load_pickle_artifact

In [12]:
bpr_calibrated = "experiments/globo/bpr_calibrated.yaml"
bpr_uncalibrated = "experiments/globo/bpr_uncalibrated.yaml"
random_rec = "experiments/globo/random_uncalibrated.yaml"

In [13]:
bpr_uncalibrated_exp = read_experiment(bpr_uncalibrated)
bpr_calibrated_exp = read_experiment(bpr_calibrated)

random_rec_exp = read_experiment(random_rec)


In [14]:
bpr_uncalibrated_exp

{'size': 's',
 'data_type': 'globo',
 'rounds': 2500,
 'num_rounds_per_eval': 250,
 'num_users': 1000,
 'model': 'bpr',
 'preference_update_rate': 0.1,
 'use_oracle_matrix': 'y',
 'exp_name': 'bpr_unbiased_uncalibrated',
 'n_trials': 10,
 'params': {'factors': 64,
  'lr': 0.0001,
  'reg_lambda': 0.001,
  'num_negatives': 5,
  'n_epochs': 7}}

In [32]:
metric_key_map = {
    "mace": "maces",
    "diver": "divergences",
    "coverage": "coverages",
    "gini": "ginis",
    "div": "diversities",
    "frags": "fragmentation",
}

def persist_metrics(exps, metric):
    """
    Collect a chosen metric for three experiments (uncalibrated, calibrated_time, calibrated_constant)
    and return a DataFrame with one column per experiment.

    Usage:
      persist_metrics(bpr_uncalibrated_exp, bpr_calibrated_time_exp, bpr_calibrated_exp, "mace")

    No file is written; the DataFrame is returned.
    """
    if not isinstance(metric, str):
        raise ValueError("metric must be a string")

    if metric not in metric_key_map:
        raise ValueError(f"Unknown metric '{metric}'. Valid keys: {list(metric_key_map.keys())}")

    metric_idx = metric_key_map[metric]

    cols = {}
    for i, exp in enumerate(exps):
        try:
            # All experiments ran for seed 42
            metrics = read_metrics_per_seed(exp, seed=42)
        except Exception as e:
            raise RuntimeError(f"read_metrics failed for experiment #{i}: {e}")

        try:
            metric_values = metrics[metric_idx]
        except Exception as e:
            raise RuntimeError(f"Could not extract metric index {metric_idx} from read_metrics output: {e}")

        ser = pd.Series(metric_values)
        col_name = None
        if isinstance(exp, dict):
            col_name = exp.get("exp_name") or exp.get("name")
        if not col_name:
            col_name = f"exp_{i}"
        cols[col_name] = ser

        # ensure we have a 'round' column that matches the row index (0..n-1)
        if "round" not in cols:
            cols["round"] = pd.Series(range(len(ser)))

    return pd.DataFrame(cols)


### Figure 2: Miscalibration

In [33]:
misc_df = persist_metrics([bpr_uncalibrated_exp, bpr_calibrated_exp, random_rec_exp], "diver")

In [34]:
misc_df

,bpr_unbiased_uncalibrated,round,bpr_unbiased_calibrated,random_uncalibrated
0,0.621445,0,0.586520,0.781200
1,0.623025,1,0.623638,0.785389
2,0.601936,2,0.638631,0.785218
3,0.602085,3,0.645139,0.763883
4,0.601556,4,0.638723,0.757338
...,...,...,...,...
2495,0.611287,2495,0.624868,0.789251
2496,0.611187,2496,0.625074,0.797242
2497,0.611074,2497,0.625074,0.768384
2498,0.611264,2498,0.625352,0.781351


In [35]:
misc_df.to_pickle(f"{reproductibiliy_path}/figure_2_globo.pkl")
misc_df.to_csv(f"{reproductibiliy_path}/figure_2_globo.csv")


### Figure 3: Catalog Coverage

In [36]:
coverage_df = persist_metrics([bpr_uncalibrated_exp, bpr_calibrated_exp, random_rec_exp], "coverage")

coverage_df.to_pickle(f"{reproductibiliy_path}/figure_3_globo.pkl")
coverage_df.to_csv(f"{reproductibiliy_path}/figure_3_globo.csv")


In [37]:
coverage_df

,bpr_unbiased_uncalibrated,round,bpr_unbiased_calibrated,random_uncalibrated
0,0.001752,0,0.001617,0.736370
1,0.002696,1,0.002156,0.742368
2,0.003302,2,0.002898,0.737314
3,0.003437,3,0.003100,0.743514
4,0.003909,4,0.003369,0.740212
...,...,...,...,...
2495,0.004852,2495,0.003639,0.737718
2496,0.004852,2496,0.003639,0.744390
2497,0.004852,2497,0.003639,0.740481
2498,0.004852,2498,0.003639,0.741559


### Figure 4: ILS

In [39]:
div_df = persist_metrics([bpr_uncalibrated_exp, bpr_calibrated_exp, random_rec_exp], "div")
div_df

,bpr_unbiased_uncalibrated,round,bpr_unbiased_calibrated,random_uncalibrated
0,0.974137,0,0.958733,0.984321
1,0.964658,1,0.964133,0.983863
2,0.957868,2,0.957244,0.983742
3,0.950679,3,0.948822,0.984084
4,0.948174,4,0.943311,0.984121
...,...,...,...,...
2495,0.949326,2495,0.943711,0.984358
2496,0.949242,2496,0.943800,0.983953
2497,0.949063,2497,0.944022,0.984484
2498,0.948989,2498,0.943911,0.983921


In [40]:

div_df.to_pickle(f"{reproductibiliy_path}/figure_4_globo.pkl")
div_df.to_csv(f"{reproductibiliy_path}/figure_4_globo.csv")



### Figure 5: Fragmentation

In [41]:
frags_df = persist_metrics([bpr_uncalibrated_exp, bpr_calibrated_exp, random_rec_exp], "frags")
frags_df

,bpr_unbiased_uncalibrated,round,bpr_unbiased_calibrated,random_uncalibrated
0,0.066485,0,0.214710,0.999510
1,0.593983,1,0.654662,0.999506
2,0.666860,2,0.737605,0.999512
3,0.697886,3,0.775618,0.999517
4,0.712056,4,0.792385,0.999507
...,...,...,...,...
2495,0.781633,2495,0.834781,0.999523
2496,0.782377,2496,0.835018,0.999517
2497,0.783146,2497,0.835259,0.999527
2498,0.784061,2498,0.835739,0.999519


In [42]:

frags_df.to_pickle(f"{reproductibiliy_path}/figure_5_globo.pkl")
frags_df.to_csv(f"{reproductibiliy_path}/figure_5_globo.csv")

